<a href="https://colab.research.google.com/github/simar-rekhi/triton/blob/neel/Nondeterminism_GPU.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
  !pip install torch

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import torch
import random
import pdb

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

kernels = {
    'cumsum': torch.cumsum,
    'sum': torch.sum,
    'matmul': torch.matmul
}

MAX_ELEMENTS = 20000000

def generate_shapes_random(num_cases=10):
    shapes = []
    for _ in range(num_cases):
        dims = random.randint(2, 4)
        shape = []
        total = 1
        for d in range(dims):
            dim_size = random.randint(512, 4096)
            total *= dim_size
            if total > MAX_ELEMENTS:
                dim_size = max(2, MAX_ELEMENTS // (total // dim_size))
                total = total // shape[-1] * dim_size if shape else dim_size
            shape.append(dim_size)
        shapes.append(tuple(shape))
    return shapes

def generate_shapes_all_dimension(max_dims=3):
    shapes = []
    for dims in range(1, max_dims + 1):
        max_per_dim = int(MAX_ELEMENTS ** (1 / dims))
        shape = []
        for _ in range(dims):
            dim_size = random.randint(2, max_per_dim)
            shape.append(dim_size)
        shapes.append(tuple(shape))
    return shapes

shapes = generate_shapes_random(10)

def generate_inputs(kernel_name):
    inputs = []
    for shape in shapes:
        size = torch.prod(torch.tensor(shape)).item()
        if kernel_name in ['sum', 'cumsum']:
            dtype = torch.float16 if size > 1_000_000 else torch.float32
            x = torch.rand(shape, dtype=dtype) * 60000 + 2
        elif kernel_name == 'matmul':
            n = random.randint(100, 500)
            m = random.randint(100, 500)
            k = random.randint(100, 500)
            a = torch.rand((n, k), dtype=torch.float32)
            b = torch.rand((k, m), dtype=torch.float32)
            inputs.append((a, b))
            continue
        inputs.append(x)
    return inputs

test_inputs = {name: generate_inputs(name) for name in kernels.keys()}

num_runs = 100

def test_nondeterminism(kernel_name, kernel_fn, inputs):
    print(f"\n--- Testing kernel: {kernel_name} ---")
    for i, input_data in enumerate(inputs):
        results = []
        for _ in range(num_runs):
            if kernel_name == 'matmul':
                a, b = input_data
                a_gpu, b_gpu = a.to('cuda'), b.to('cuda')
                out = kernel_fn(a_gpu, b_gpu)
            else:
                x = input_data.to('cuda')
                if kernel_name == 'cumsum':
                    out = kernel_fn(x, dim=0)
                else:
                    out = kernel_fn(x)
            results.append(out.detach().cpu())
            if kernel_name == 'matmul':
                del a_gpu, b_gpu, out
            else:
                del x, out
            torch.cuda.empty_cache()
        identical = all(torch.allclose(results[0], r, rtol=1e-7, atol=1e-9) for r in results[1:])
        status = "Deterministic" if identical else "Non-deterministic"
        if kernel_name == 'matmul':
            print(f"Test {i+1:2d} | shape {input_data[0].shape} x {input_data[1].shape} | dtype {input_data[0].dtype} | {status}")
        else:
            print(f"Test {i+1:2d} | shape {input_data.shape} | dtype {input_data.dtype} | {status}")

for name, fn in kernels.items():
    test_nondeterminism(name, fn, test_inputs[name])


--- Testing kernel: cumsum ---
Test  1 | shape torch.Size([3325, 3729]) | dtype torch.float16 | Deterministic
Test  2 | shape torch.Size([1094, 2172, 8, 2]) | dtype torch.float16 | Deterministic
Test  3 | shape torch.Size([2862, 2728, 2]) | dtype torch.float16 | Deterministic
Test  4 | shape torch.Size([1222, 1574, 10]) | dtype torch.float16 | Deterministic
Test  5 | shape torch.Size([2347, 1929]) | dtype torch.float16 | Deterministic
Test  6 | shape torch.Size([2622, 3678, 2, 6]) | dtype torch.float16 | Deterministic
Test  7 | shape torch.Size([3829, 648, 8, 2]) | dtype torch.float16 | Deterministic
Test  8 | shape torch.Size([1451, 2166, 6]) | dtype torch.float16 | Deterministic
Test  9 | shape torch.Size([2271, 2066]) | dtype torch.float16 | Deterministic
Test 10 | shape torch.Size([3281, 945]) | dtype torch.float16 | Deterministic

--- Testing kernel: sum ---
Test  1 | shape torch.Size([3325, 3729]) | dtype torch.float16 | Deterministic
Test  2 | shape torch.Size([1094, 2172, 8, 2